### Đọc dữ liệu đã làm sạch

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/depression_severity_cleaned.csv')

print("Số dòng:", len(df))
print(df['label'].value_counts())

Số dòng: 3531
label
minimum     2566
moderate     394
mild         290
severe       281
Name: count, dtype: int64


### Kiểm tra text trùng nhưng khác nhãn

In [3]:
dup_text = df[df.duplicated(subset=['text'], keep=False)]

print("Số dòng có text trùng với dòng khác:", len(dup_text))

conflict = dup_text.groupby('text')['label'].nunique()
conflict = conflict[conflict > 1]

print("Số nhóm text trùng nhưng nhãn khác nhau:", len(conflict))
print(conflict)

Số dòng có text trùng với dòng khác: 0
Số nhóm text trùng nhưng nhãn khác nhau: 0
Series([], Name: label, dtype: int64)


### 3 Kiểm tra near-duplicate (cosine similarity)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(max_features=2000)
tfidf_matrix = vectorizer.fit_transform(df['text'])

sim_matrix = cosine_similarity(tfidf_matrix)
np.fill_diagonal(sim_matrix, 0)

threshold = 0.9
pairs = np.argwhere(sim_matrix > threshold)
pairs = pairs[pairs[:, 0] < pairs[:, 1]]

print("Số cặp near-duplicate (similarity >", threshold, "):", len(pairs))

Số cặp near-duplicate (similarity > 0.9 ): 13


### 4 Xem chi tiết các cặp near-duplicate

In [5]:
for i, j in pairs:
    print("=" * 60)
    print(f"Similarity: {sim_matrix[i, j]:.3f}")
    print(f"[{df.iloc[i]['label']}] {df.iloc[i]['text'][:100]}")
    print(f"[{df.iloc[j]['label']}] {df.iloc[j]['text'][:100]}")

Similarity: 0.975
[minimum] I'm looking for anyone who is interested in taking a 10 minute survey, with the chance to win a £100
[minimum] Hello lovely people! I'm looking for anyone who is interested in taking a 10 minute survey, with the
Similarity: 0.945
[minimum] I go to the VA and I see people who need it more than me. I make a good living and only want to get 
[minimum] I make a good living and only want to get on with life. Plus I know filing a claim at the VA is humi
Similarity: 0.992
[minimum] If you have a survey you would like to share with us, you may do so here, please use the following f
[minimum] If you have a survey you would like to share with us, you may do so here, please use the following f
Similarity: 0.976
[minimum] Hello, You are invited to complete a survey for a WMU psychology department research project designe
[minimum] Hello, You are invited to complete a survey for a WMU psychology department research project designe
Similarity: 0.955
[minimum] Most diagnos

### 5. Loại near-duplicate (giữ 1 bài mỗi cặp)

In [6]:
n_before = len(df)

# Lấy chỉ số các dòng cần loại (giữ i, loại j trong mỗi cặp)
idx_to_drop = set(df.index[j] for i, j in pairs)

df = df.drop(index=idx_to_drop).reset_index(drop=True)
n_after = len(df)

print(f"Trước khi loại: {n_before} dòng")
print(f"Sau khi loại:   {n_after} dòng")
print(f"Đã loại:        {n_before - n_after} dòng near-duplicate")

Trước khi loại: 3531 dòng
Sau khi loại:   3519 dòng
Đã loại:        12 dòng near-duplicate


### 3.6 Kiểm tra user trùng lặp

In [7]:
print("Các cột hiện có trong dataset:", list(df.columns))
print("\n→ Dataset không có cột user_id, nên không thể kiểm tra")
print("  hiện tượng cùng 1 user xuất hiện ở cả train và test.")
print("  Ghi chú này sẽ đưa vào phần hạn chế của báo cáo.")

Các cột hiện có trong dataset: ['text', 'label']

→ Dataset không có cột user_id, nên không thể kiểm tra
  hiện tượng cùng 1 user xuất hiện ở cả train và test.
  Ghi chú này sẽ đưa vào phần hạn chế của báo cáo.


### 3.7 Tổng kết và lưu file

In [8]:
print("=" * 50)
print("TỔNG KẾT DATA LEAKAGE CHECK")
print("=" * 50)
print("Text trùng nhưng khác nhãn: 0 (không có)")
print("Near-duplicate đã loại:     12 dòng")
print(f"Số dòng còn lại:            {len(df)}")

print("\nPhân bố nhãn sau Bước 3:")
print(df['label'].value_counts())

df.to_csv('../data/processed/depression_severity_no_leakage.csv', index=False)
print("\nĐã lưu: data/processed/depression_severity_no_leakage.csv")

TỔNG KẾT DATA LEAKAGE CHECK
Text trùng nhưng khác nhãn: 0 (không có)
Near-duplicate đã loại:     12 dòng
Số dòng còn lại:            3519

Phân bố nhãn sau Bước 3:
label
minimum     2555
moderate     393
mild         290
severe       281
Name: count, dtype: int64

Đã lưu: data/processed/depression_severity_no_leakage.csv
